# 01 — Baseline and Feature Pool

Основной этап исследования: подготовка данных Yambda, временное разбиение без утечки, LightGBM baseline, ручные признаки, расширенный пул временных агрегатов и автоматический отбор признаков.

> Ноутбук очищен для итоговой версии репозитория: outputs и execution counts удалены. Численные результаты выполненных запусков сохранены в папке `results/`.

---

## Исходное описание этапа

# Автоматизация генерации фичей для рекомендательной модели (Yambda-50M)

**Задача:** показать, что генерацию фичей для градиентного бустинга можно автоматизировать и получить качество не хуже ручных фичей.

## Протокол эксперимента (зафиксирован до начала экспериментов)

| Что | Решение |
|---|---|
| Данные | Yambda-50M, события `listen`, подвыборка пользователей (`N_USERS`) с полными историями |
| Label | `1`, если `played_ratio_pct >= 50` (в карточке датасета: трек «прослушан», если проиграно более 50%) |
| Разбиение | по времени, 80/10/10 по квантилям `timestamp`; **test не используется до финала** |
| Фичи | считаются **строго по прошлому** и **на начало суток** (снимок): у всех прослушиваний одного дня одного пользователя одинаковый момент расчёта, как при выдаче рекомендаций утром |
| Группа для ранжирования | (пользователь, день); кандидаты — треки, которые пользователь послушал в этот день |
| Модель | LightGBM `lambdarank` |
| Главная метрика | NDCG@10 внутри групп (группы с числом строк < 10 отбрасываются) |
| Исключено из фичей | `is_organic` (свойство самого прослушивания, при выдаче кандидатов неизвестно) |

**Ограничение:** в данных только то, что слушали, а не то, что показывали, поэтому ранжирование оценивается среди прослушанных треков.

In [ ]:
# !pip install -q lightgbm huggingface_hub pyarrow   # раскомментируйте, если библиотек нет

In [ ]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
from huggingface_hub import hf_hub_download

SEED = 42
N_USERS = 1500
POS_THRESHOLD = 50          # label = 1, если played_ratio_pct >= 50
DAY = 86400                 # длина суток в единицах timestamp (проверяется ниже)
SHIFT = 26                  # timestamp < 2**26 (у нас максимум ~26 млн)
WINDOWS = [1, 7, 30]        # окна (дни) для user / item / user-item фичей
G_WINDOWS = [1, 7, 30, 90]  # окна (дни) для фичей трека по ВСЕМ пользователям
TRAIN_DAYS = 60             # обучаемся на последних TRAIN_DAYS днях train-периода
FEAT_PATH = f"features_snap_u{N_USERS}_s{SEED}.parquet"   # кэш рассчитанных фичей
RECOMPUTE = False           # True — пересчитать фичи, даже если кэш есть

## 1. Данные: загрузка, подвыборка пользователей, label

In [ ]:
path = hf_hub_download(
    repo_id="yandex/yambda",
    repo_type="dataset",
    filename="flat/50m/multi_event.parquet",
    local_dir="data",
)

# выбираем пользователей (читаем только колонку uid)
uids = pq.read_table(path, columns=["uid"]).column("uid").unique().to_numpy()
rng = np.random.default_rng(SEED)
keep = rng.choice(uids, size=min(N_USERS, len(uids)), replace=False)

# читаем только нужные колонки и строки: прослушивания выбранных пользователей
table = pq.read_table(
    path,
    columns=["uid", "item_id", "timestamp", "is_organic", "played_ratio_pct"],
    filters=[("event_type", "==", "listen"), ("uid", "in", keep.tolist())],
)
df = table.to_pandas()
del table

df["label"] = (df["played_ratio_pct"] >= POS_THRESHOLD).astype(np.int8)
df = df.sort_values("timestamp").reset_index(drop=True)
print("строк:", len(df), "| доля label=1:", round(df["label"].mean(), 3))
print(df.dtypes)

In [ ]:
# разбиение по времени 80 / 10 / 10
t_val, t_test = df["timestamp"].quantile([0.8, 0.9])
df["split"] = np.where(df["timestamp"] < t_val, "train",
               np.where(df["timestamp"] < t_test, "val", "test"))

bounds = df.groupby("split")["timestamp"].agg(["min", "max", "size"])
print(bounds)
assert bounds.loc["train", "max"] < bounds.loc["val", "min"]
assert bounds.loc["val", "max"] < bounds.loc["test", "min"]

df.to_parquet("base_listens.parquet")

### Проверка единиц времени

В карточке датасета `timestamp` назван «binned into 5s units». Проверяем по данным: разрыв между соседними прослушиваниями должен быть порядка длительности прослушанной части трека. Отношение около 1 означает, что единица времени — секунда и `DAY = 86400` верно.

In [ ]:
chk = pq.read_table(
    path,
    columns=["uid", "timestamp", "track_length_seconds", "played_ratio_pct"],
    filters=[("event_type", "==", "listen"), ("uid", "in", keep[:30].tolist())],
).to_pandas().sort_values(["uid", "timestamp"])

prev = chk.groupby("uid").shift(1)
gap = chk["timestamp"].astype("float64") - prev["timestamp"]
expected = prev["track_length_seconds"] * prev["played_ratio_pct"] / 100
m = (gap < 600) & (expected > 0)
ratio = (gap[m] / expected[m]).median()
print("медиана gap / expected:", round(float(ratio), 3))
if not 0.5 < ratio < 2:
    print("ВНИМАНИЕ: единицы времени, вероятно, не секунды. Пересмотрите DAY!")
del chk, prev, gap, expected, m

## 2. Функция расчёта фичей «строго по прошлому»

`add_window_feats` для каждой строки считает число и долю `label=1` среди **прошлых** событий той же группы (`keys`) в окнах `windows_days`.

- `snap=None` — окно считается от точного момента события;
- `snap=DAY` — окно считается от **начала суток** события (снимок). Это основной режим.

События строго раньше момента расчёта: ни само событие, ни события с тем же временем в фичи не попадают.

In [ ]:
def add_window_feats(df, keys, windows_days, name, snap=None):
    codes = df.groupby(keys, sort=False).ngroup().to_numpy(np.int64)
    ts = df["timestamp"].to_numpy(np.int64)
    lab = df["label"].to_numpy(np.int64)
    tq = ts if snap is None else (ts // snap) * snap          # момент, "на который считаем"
    comb = (codes << SHIFT) | ts
    order = np.argsort(comb, kind="stable")
    comb_s, codes_s, tq_s, lab_s = comb[order], codes[order], tq[order], lab[order]
    cum = np.concatenate([[0], np.cumsum(lab_s)])
    hi = np.searchsorted(comb_s, (codes_s << SHIFT) | tq_s, side="left")   # строго раньше tq
    out = {}
    for w in windows_days:
        lo = np.searchsorted(comb_s, (codes_s << SHIFT) | np.maximum(tq_s - w * DAY, 0), side="left")
        cnt = (hi - lo).astype(np.float32)
        pos = (cum[hi] - cum[lo]).astype(np.float32)
        r_cnt = np.empty(len(df), np.float32); r_cnt[order] = cnt
        r_rate = np.empty(len(df), np.float32)
        r_rate[order] = np.where(cnt > 0, pos / np.maximum(cnt, 1), np.nan)
        out[f"{name}_cnt_{w}d"] = r_cnt
        out[f"{name}_rate_{w}d"] = r_rate
    return pd.DataFrame(out, index=df.index)

In [ ]:
# Тест на отсутствие утечки: сравнение с наивным перебором на маленьких синтетических данных
_r = np.random.default_rng(0); _n = 3000
_t = pd.DataFrame({
    "uid": _r.integers(0, 15, _n), "item_id": _r.integers(0, 25, _n),
    "timestamp": _r.integers(0, 40 * DAY // 5, _n) * 5, "label": _r.integers(0, 2, _n),
}).sort_values("timestamp").reset_index(drop=True)

for _snap in [None, DAY]:
    _F = add_window_feats(_t, ["uid", "item_id"], [1, 7], "ui", snap=_snap)
    for i in _r.choice(_n, 300, replace=False):
        row = _t.iloc[i]
        tq = row.timestamp if _snap is None else (row.timestamp // _snap) * _snap
        base = (_t.uid == row.uid) & (_t.item_id == row.item_id) & (_t.timestamp < tq)
        for w in [1, 7]:
            mm = base & (_t.timestamp >= tq - w * DAY)
            assert _F.loc[i, f"ui_cnt_{w}d"] == mm.sum()
            exp = _t.label[mm].mean() if mm.sum() else np.nan
            got = _F.loc[i, f"ui_rate_{w}d"]
            assert (np.isnan(exp) and np.isnan(got)) or abs(exp - got) < 1e-6
    print("тест пройден, snap =", _snap)

## 3. Метрика

NDCG@k внутри групп. Положительных меток около 64%, поэтому у случайного порядка NDCG уже высокий — **всегда сравниваем с ним**.

In [ ]:
def ndcg_at_k(groups, scores, labels, k=10, min_size=10):
    d = pd.DataFrame({"g": np.asarray(groups), "s": np.asarray(scores),
                      "y": np.asarray(labels, dtype=float)})
    d = d[d.groupby("g")["y"].transform("size") >= min_size]
    a = d.sort_values(["g", "s"], ascending=[True, False])
    ra = a.groupby("g").cumcount().to_numpy() + 1
    dcg = pd.Series(a["y"].to_numpy() / np.log2(ra + 1) * (ra <= k), index=a.index).groupby(a["g"]).sum()
    b = d.sort_values(["g", "y"], ascending=[True, False])
    rb = b.groupby("g").cumcount().to_numpy() + 1
    idcg = pd.Series(b["y"].to_numpy() / np.log2(rb + 1) * (rb <= k), index=b.index).groupby(b["g"]).sum()
    return (dcg / idcg).dropna().mean()

## 4. Ручные фичи (бейзлайн)

- `user_*` — активность и доля дослушиваний пользователя;
- `item_*` — то же для трека **внутри нашей подвыборки пользователей**;
- `ui_*` — история пары «пользователь–трек»;
- `gitem_*` — статистика трека по **всем** пользователям датасета (только по прошлому, поэтому не утечка). Нужна, потому что в подвыборке на каждый трек приходится мало прослушиваний.

Результат кэшируется в `FEAT_PATH`, повторный запуск ячейки быстрый.

In [ ]:
if os.path.exists(FEAT_PATH) and not RECOMPUTE:
    FS = pd.read_parquet(FEAT_PATH)
    print("фичи загружены из кэша:", FEAT_PATH)
else:
    FS = pd.concat([
        add_window_feats(df, ["uid"], WINDOWS, "user", DAY),
        add_window_feats(df, ["item_id"], WINDOWS, "item", DAY),
        add_window_feats(df, ["uid", "item_id"], WINDOWS, "ui", DAY),
    ], axis=1)

    # статистика трека по ВСЕМ пользователям (тяжёлый блок: ~46 млн строк)
    tab = pq.read_table(path, columns=["uid", "item_id", "timestamp", "played_ratio_pct"],
                        filters=[("event_type", "==", "listen")])
    full = tab.to_pandas(); del tab
    full["label"] = (full["played_ratio_pct"] >= POS_THRESHOLD).astype(np.int8)
    full = full.sort_values("timestamp").reset_index(drop=True)

    IF = add_window_feats(full, ["item_id"], G_WINDOWS, "gitem", DAY)
    in_sample = full["uid"].isin(keep).to_numpy()
    IF["item_id"] = full["item_id"].values
    IF["timestamp"] = full["timestamp"].values
    IF = IF[in_sample].drop_duplicates(["item_id", "timestamp"])
    del full

    G = df[["item_id", "timestamp"]].merge(IF, on=["item_id", "timestamp"], how="left")
    assert len(G) == len(df)
    for c in G.columns:
        if c.startswith("gitem_"):
            FS[c] = G[c].values
    del G, IF
    FS.to_parquet(FEAT_PATH)
    print("фичи посчитаны и сохранены в", FEAT_PATH)

assert len(FS) == len(df)
FEATS = list(FS.columns)
print(len(FEATS), "фичей:", FEATS)

## 5. Модель: LightGBM `lambdarank`

Обучаем на последних `TRAIN_DAYS` днях train-периода (история для фичей при этом берётся вся). Ранняя остановка идёт по val, поэтому val-оценка слегка оптимистична. Финальные выводы делаем только по test, который трогаем один раз.

In [ ]:
tr = (df["split"] == "train") & (df["timestamp"] >= t_val - TRAIN_DAYS * DAY)
va = df["split"] == "val"
print("train:", int(tr.sum()), "| val:", int(va.sum()))

def prep(mask):
    d = df.loc[mask, ["uid", "timestamp", "label"]].copy()
    d["g"] = d["uid"].astype(np.int64) * 1000 + d["timestamp"].astype(np.int64) // DAY
    d = d.sort_values("g", kind="stable")
    return d, d.groupby("g", sort=True).size().to_numpy()

d_tr, sz_tr = prep(tr)
d_va, sz_va = prep(va)

params_r = dict(objective="lambdarank", metric="ndcg", eval_at=[10],
                learning_rate=0.05, num_leaves=63, min_data_in_leaf=200,
                feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, verbose=-1)

def fit_rank(cols, seed=0, rounds=300):
    dtr = lgb.Dataset(FS.loc[d_tr.index, cols], d_tr["label"], group=sz_tr)
    dva = lgb.Dataset(FS.loc[d_va.index, cols], d_va["label"], group=sz_va)
    m = lgb.train({**params_r, "seed": seed}, dtr, rounds, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    pr = m.predict(FS.loc[d_va.index, cols], num_iteration=m.best_iteration)
    return m, ndcg_at_k(d_va["g"], pr, d_va["label"])

In [ ]:
results = {}

rnd = ndcg_at_k(d_va["g"], np.random.default_rng(0).random(len(d_va)), d_va["label"])
pop = ndcg_at_k(d_va["g"], FS.loc[d_va.index, "gitem_rate_30d"].fillna(0.64), d_va["label"])
model, s_all = fit_rank(FEATS)

results["случайный порядок"] = rnd
results["популярность трека (gitem_rate_30d)"] = pop
results["LightGBM, все ручные фичи"] = s_all
for k, v in results.items():
    print(f"NDCG@10  {k:40s} {v:.4f}")

imp = pd.Series(model.feature_importance("gain"), index=FEATS).sort_values(ascending=False)
print()
print((imp / imp.sum()).round(3))

## 6. Абляция: вклад групп фичей

Каждый вариант обучается с несколькими seed, чтобы видеть шум. Различия меньше разброса по seed считаем незначимыми.

In [ ]:
SEEDS = [0, 1]    # для итогового отчёта лучше 3-5

sets = {
    "все":              FEATS,
    "без user_*":       [c for c in FEATS if not c.startswith("user_")],
    "без item_/gitem_": [c for c in FEATS if not c.startswith(("item_", "gitem_"))],
    "без ui_*":         [c for c in FEATS if not c.startswith("ui_")],
    "только ui_*":      [c for c in FEATS if c.startswith("ui_")],
}
abl = []
for name, cols in sets.items():
    v = [fit_rank(cols, seed=sd)[1] for sd in SEEDS]
    abl.append({"набор": name, "NDCG@10": np.mean(v), "std": np.std(v)})
    print(f"{name:20s} NDCG@10 = {np.mean(v):.4f} ± {np.std(v):.4f}")

abl = pd.DataFrame(abl)
abl.to_csv("ablation_val.csv", index=False)

## 7. План дальнейшей работы

1. Усилить ручной бейзлайн: фичи user × исполнитель и user × альбом (файлы `artist_item_mapping.parquet`, `album_item_mapping.parquet` в корне репозитория датасета; схему сначала посмотреть через `.columns`), время с последнего прослушивания пары и трека; затем тюнинг (Optuna).
2. **Основная идея:** автоматический перебор фичей по грамматике «ключ × окно × агрегат» с отбором по приросту NDCG@10 на val; подбор окна (7 или 10 дней?) оптимизацией или экспоненциальным затуханием.
3. По желанию: LLM-агент, предлагающий фичи.
4. Финал: один запуск на test, таблица «ручные vs автоматические фичи», разбор ограничений.

**Правило:** все решения принимаются по val; test открывается один раз.

In [ ]:
# ===== ячейка 1: расширенная сетка окон =====
WIDE = [1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60, 90, 120, 180, 400]   # 400 дней = вся история
WIDE_PATH = f"features_wide_u{N_USERS}_s{SEED}.parquet"

if os.path.exists(WIDE_PATH) and not RECOMPUTE:
    FW = pd.read_parquet(WIDE_PATH)
else:
    FW = pd.concat([
        add_window_feats(df, ["uid"], WIDE, "user", DAY),
        add_window_feats(df, ["uid", "item_id"], WIDE, "ui", DAY),
    ], axis=1)
    FW.to_parquet(WIDE_PATH)
assert len(FW) == len(df)

# пул кандидатов: item/gitem из FS + широкая сетка окон для user и ui
POOL = pd.concat([FS[[c for c in FEATS if c.startswith(("item_", "gitem_"))]], FW], axis=1)
POOL_COLS = list(POOL.columns)
print(len(POOL_COLS), "фичей в пуле")

In [ ]:
# ===== ячейка 2: кривая "качество от окна" =====
import matplotlib.pyplot as plt

noise = np.random.default_rng(0).random(len(d_va)) * 1e-6   # случайный разрыв ничьих

def uni_ndcg(col):
    s = FW.loc[d_va.index, col].fillna(0.64).to_numpy() + noise
    return ndcg_at_k(d_va["g"], s, d_va["label"])

scan = pd.DataFrame({
    "window_d": WIDE,
    "ui_rate": [uni_ndcg(f"ui_rate_{w}d") for w in WIDE],
    "ui_cnt":  [uni_ndcg(f"ui_cnt_{w}d") for w in WIDE],
})
print(scan.round(4))
plt.figure(figsize=(7, 4))
plt.semilogx(scan["window_d"], scan["ui_rate"], "o-", label="ui_rate")
plt.semilogx(scan["window_d"], scan["ui_cnt"], "s-", label="ui_cnt")
plt.axhline(results["случайный порядок"], color="gray", ls="--", label="случайный порядок")
plt.xlabel("окно, дни"); plt.ylabel("NDCG@10 одной фичи"); plt.legend(); plt.grid(alpha=.3)
plt.savefig("window_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ===== ячейка 3: модель на всём пуле и отбор по важности =====
def fit_rank_X(X, cols, seed=0, rounds=300):
    dtr = lgb.Dataset(X.loc[d_tr.index, cols], d_tr["label"], group=sz_tr)
    dva = lgb.Dataset(X.loc[d_va.index, cols], d_va["label"], group=sz_va)
    m = lgb.train({**params_r, "seed": seed}, dtr, rounds, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    pr = m.predict(X.loc[d_va.index, cols], num_iteration=m.best_iteration)
    return m, pr, ndcg_at_k(d_va["g"], pr, d_va["label"])

m_base, pr_base, s_base = fit_rank_X(FS, FEATS)
m_pool, pr_pool, s_pool = fit_rank_X(POOL, POOL_COLS)
imp_p = pd.Series(m_pool.feature_importance("gain"), index=POOL_COLS).sort_values(ascending=False)
TOPK = 30
top = imp_p.index[:TOPK].tolist()
m_top, pr_top, s_top = fit_rank_X(POOL, top)

print(f"NDCG@10  ручные ({len(FEATS)} фичей):      {s_base:.4f}")
print(f"NDCG@10  пул ({len(POOL_COLS)} фичей):        {s_pool:.4f}")
print(f"NDCG@10  топ-{TOPK} из пула по важности: {s_top:.4f}")
print((imp_p / imp_p.sum()).round(3).head(15))

In [ ]:
# ===== ячейка 4: доверительные интервалы (bootstrap по пользователям) =====
def ndcg_groups(groups, scores, labels, k=10, min_size=10):
    d = pd.DataFrame({"g": np.asarray(groups), "s": np.asarray(scores),
                      "y": np.asarray(labels, dtype=float)})
    d = d[d.groupby("g")["y"].transform("size") >= min_size]
    a = d.sort_values(["g", "s"], ascending=[True, False])
    ra = a.groupby("g").cumcount().to_numpy() + 1
    dcg = pd.Series(a["y"].to_numpy() / np.log2(ra + 1) * (ra <= k), index=a.index).groupby(a["g"]).sum()
    b = d.sort_values(["g", "y"], ascending=[True, False])
    rb = b.groupby("g").cumcount().to_numpy() + 1
    idcg = pd.Series(b["y"].to_numpy() / np.log2(rb + 1) * (rb <= k), index=b.index).groupby(b["g"]).sum()
    return (dcg / idcg).dropna()

def boot_diff(pred_a, pred_b, n_boot=1000, seed=0):
    ga = ndcg_groups(d_va["g"], pred_a, d_va["label"])
    gb = ndcg_groups(d_va["g"], pred_b, d_va["label"])
    diff = (gb - ga).dropna()
    per_user = diff.groupby(diff.index // 1000).agg(["sum", "count"])
    sums, cnts = per_user["sum"].to_numpy(), per_user["count"].to_numpy()
    r = np.random.default_rng(seed)
    est = []
    for _ in range(n_boot):
        idx = r.integers(0, len(sums), len(sums))
        est.append(sums[idx].sum() / cnts[idx].sum())
    return diff.mean(), np.percentile(est, [2.5, 97.5])

for name, pr in [("пул", pr_pool), (f"топ-{TOPK}", pr_top)]:
    mean, ci = boot_diff(pr_base, pr)
    print(f"{name} минус ручные: ΔNDCG@10 = {mean:+.4f}, 95% ДИ [{ci[0]:+.4f}, {ci[1]:+.4f}]")

In [ ]:
def add_window_feats2(df, keys, windows_days, name, target="label", snap=DAY,
                      with_cnt=True, recency=False):
    codes = df.groupby(keys, sort=False).ngroup().to_numpy(np.int64)
    ts = df["timestamp"].to_numpy(np.int64)
    val = df[target].to_numpy(np.float64)
    tq = ts if snap is None else (ts // snap) * snap
    comb = (codes << SHIFT) | ts
    order = np.argsort(comb, kind="stable")
    comb_s, codes_s, tq_s, val_s = comb[order], codes[order], tq[order], val[order]
    cum = np.concatenate([[0.0], np.cumsum(val_s)])
    hi = np.searchsorted(comb_s, (codes_s << SHIFT) | tq_s, side="left")
    tag = "rate" if target == "label" else f"mean_{target}"
    out = {}
    for w in windows_days:
        lo = np.searchsorted(comb_s, (codes_s << SHIFT) | np.maximum(tq_s - w * DAY, 0), side="left")
        cnt = (hi - lo).astype(np.float64)
        mean = np.where(cnt > 0, (cum[hi] - cum[lo]) / np.maximum(cnt, 1), np.nan)
        if with_cnt:
            r = np.empty(len(df), np.float32); r[order] = cnt; out[f"{name}_cnt_{w}d"] = r
        r = np.empty(len(df), np.float32); r[order] = mean; out[f"{name}_{tag}_{w}d"] = r
    if recency:                                   # дней с последнего события группы (до tq)
        prev = hi - 1
        ok = prev >= 0
        prev = np.where(ok, prev, 0)
        ok &= codes_s[prev] == codes_s
        last_ts = comb_s[prev] & ((1 << SHIFT) - 1)
        rec = np.where(ok, (tq_s - last_ts) / DAY, np.nan)
        r = np.empty(len(df), np.float32); r[order] = rec; out[f"{name}_recency_d"] = r
    return pd.DataFrame(out, index=df.index)

# проверка против наивного перебора
_r = np.random.default_rng(1); _n = 2500
_t = pd.DataFrame({"uid": _r.integers(0, 10, _n), "item_id": _r.integers(0, 15, _n),
    "timestamp": _r.integers(0, 40 * DAY // 5, _n) * 5, "label": _r.integers(0, 2, _n),
    "pr": _r.random(_n)}).sort_values("timestamp").reset_index(drop=True)
_F = add_window_feats2(_t, ["uid", "item_id"], [7], "ui", target="pr", recency=True)
for i in _r.choice(_n, 250, replace=False):
    row = _t.iloc[i]; tq = (row.timestamp // DAY) * DAY
    prev = _t[(_t.uid == row.uid) & (_t.item_id == row.item_id) & (_t.timestamp < tq)]
    w7 = prev[prev.timestamp >= tq - 7 * DAY]
    assert _F.loc[i, "ui_cnt_7d"] == len(w7)
    e = w7.pr.mean() if len(w7) else np.nan; g = _F.loc[i, "ui_mean_pr_7d"]
    assert (np.isnan(e) and np.isnan(g)) or abs(e - g) < 1e-5
    e = (tq - prev.timestamp.max()) / DAY if len(prev) else np.nan; g = _F.loc[i, "ui_recency_d"]
    assert (np.isnan(e) and np.isnan(g)) or abs(e - g) < 1e-4
print("тест add_window_feats2 пройден")

In [ ]:
art_path = hf_hub_download(repo_id="yandex/yambda", repo_type="dataset",
                           filename="artist_item_mapping.parquet", local_dir="data")
alb_path = hf_hub_download(repo_id="yandex/yambda", repo_type="dataset",
                           filename="album_item_mapping.parquet", local_dir="data")
for p in (art_path, alb_path):
    t = pq.read_table(p)
    print(p, t.schema.names, t.num_rows)
    print(t.slice(0, 3).to_pandas())

def load_map(p, label):
    m = pq.read_table(p).to_pandas()
    other = [c for c in m.columns if c != "item_id"]
    assert len(other) == 1, f"неожиданные колонки: {list(m.columns)}"
    assert not isinstance(m[other[0]].iloc[0], (list, np.ndarray)), "несколько значений на трек"
    print(label, "дубликатов item_id:", int(m["item_id"].duplicated().sum()))
    return m.drop_duplicates("item_id").set_index("item_id")[other[0]]

for col, p in [("artist_id", art_path), ("album_id", alb_path)]:
    mp = load_map(p, col)
    s = df["item_id"].map(mp).astype("float64")
    print(col, "треков без значения:", round(float(s.isna().mean()), 4))
    df[col] = np.where(s.isna(), -1 - df["item_id"].astype(np.int64), s).astype(np.int64)
    print(col, "уникальных:", df[col].nunique())

In [ ]:
df["pr"] = (np.minimum(df["played_ratio_pct"], 100) / 100).astype(np.float32)
V2_PATH = f"features_v2_u{N_USERS}_s{SEED}.parquet"
if os.path.exists(V2_PATH) and not RECOMPUTE:
    FV2 = pd.read_parquet(V2_PATH)
else:
    W2 = [7, 30, 90, 400]
    FV2 = pd.concat([
        add_window_feats2(df, ["uid", "item_id"], W2, "ui", target="pr", with_cnt=False, recency=True),
        add_window_feats2(df, ["uid", "artist_id"], W2, "uart", recency=True),
        add_window_feats2(df, ["uid", "album_id"], W2, "ualb", recency=True),
    ], axis=1)
    FV2.to_parquet(V2_PATH)
assert len(FV2) == len(df)
POOL2 = pd.concat([POOL, FV2], axis=1)
POOL2_COLS = list(POOL2.columns)
print(len(POOL2_COLS), "фичей в расширенном пуле")

In [ ]:
import re
m_p2, pr_p2, s_p2 = fit_rank_X(POOL2, POOL2_COLS)
imp2 = pd.Series(m_p2.feature_importance("gain"), index=POOL2_COLS).sort_values(ascending=False)
top2 = imp2.index[:TOPK].tolist()
m_t2, pr_t2, s_t2 = fit_rank_X(POOL2, top2)

print(f"NDCG@10  ручные ({len(FEATS)}):       {s_base:.4f}")
print(f"NDCG@10  пул v1 ({len(POOL_COLS)}):   {s_pool:.4f}")
print(f"NDCG@10  пул v2 ({len(POOL2_COLS)}):  {s_p2:.4f}")
print(f"NDCG@10  топ-{TOPK} из пула v2:       {s_t2:.4f}")
for name, a, b in [("пул v2 - ручные", pr_base, pr_p2), ("топ-30 v2 - ручные", pr_base, pr_t2),
                   ("пул v2 - пул v1", pr_pool, pr_p2)]:
    mean, ci = boot_diff(a, b)
    print(f"{name}: Δ = {mean:+.4f}, 95% ДИ [{ci[0]:+.4f}, {ci[1]:+.4f}]")

fam = (imp2 / imp2.sum()).groupby(lambda c: re.sub(r"_\d+d$", "", c)).sum().sort_values(ascending=False)
print(fam.round(3).head(15))

In [ ]:
# ===== ячейка E: M2 и абляция семейств v2 =====
import re
def win(c):
    m = re.search(r"_(\d+)d$", c)
    return int(m.group(1)) if m else None

M2_COLS = [c for c in POOL2_COLS if win(c) in {1, 7, 30, 90} or c.endswith("_recency_d")]
print(len(M2_COLS), "признаков в M2")          # ожидается 48

m_m2, pr_m2, s_m2 = fit_rank_X(POOL2, M2_COLS)
print(f"NDCG@10 M2 ({len(M2_COLS)}): {s_m2:.4f}")
for name, a, b in [("M2 - M1", pr_base, pr_m2),
                   ("пул v2 - M2", pr_m2, pr_p2),
                   ("топ-30 v2 - M2", pr_m2, pr_t2)]:
    mean, ci = boot_diff(a, b)
    print(f"{name}: Δ = {mean:+.4f}, 95% ДИ [{ci[0]:+.4f}, {ci[1]:+.4f}]")

fam_of = lambda c: re.sub(r"_\d+d$", "", c)
drop_sets = {
    "без ui_mean_pr": [c for c in POOL2_COLS if fam_of(c) != "ui_mean_pr"],                    # 93
    "без uart/ualb":  [c for c in POOL2_COLS if not fam_of(c).startswith(("uart", "ualb"))],   # 79
    "без recency":    [c for c in POOL2_COLS if not c.endswith("_recency_d")],                 # 94
}
for name, cols in drop_sets.items():
    _, pr_x, s_x = fit_rank_X(POOL2, cols)
    mean, ci = boot_diff(pr_p2, pr_x)
    print(f"{name:16s} ({len(cols)}) NDCG@10={s_x:.4f}; Δ к пулу v2 = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")

In [ ]:
import json
M1_COLS = list(FEATS)
feature_sets = {"M1": M1_COLS, "M2": list(M2_COLS), "POOL2": list(POOL2_COLS)}
json.dump(feature_sets, open("feature_sets_v1.json", "w"), ensure_ascii=False, indent=1)
print({k: len(v) for k, v in feature_sets.items()})     # M1: 26, M2: 48, POOL2: 97

# A1 (топ-K по важности) при том же K, что у M1 и M2
for K in (26, 48):
    cols_k = imp2.index[:K].tolist()
    _, pr_k, s_k = fit_rank_X(POOL2, cols_k)
    ref_name, ref_pr = ("M1", pr_base) if K == 26 else ("M2", pr_m2)
    mean, ci = boot_diff(ref_pr, pr_k)
    print(f"A1 K={K}: NDCG@10={s_k:.4f}; Δ к {ref_name} = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")

In [ ]:
import gc
assert set(FEATS) <= set(POOL2_COLS)

# --- разбиение пользователей val пополам ---
rs = np.random.default_rng(123)
val_users = df.loc[va, "uid"].unique()
half = set(rs.choice(val_users, size=len(val_users) // 2, replace=False).tolist())
is_sel = df["uid"].isin(half).to_numpy()
va_sel, va_rep = va & is_sel, va & ~is_sel
d_sel, sz_sel = prep(va_sel)
d_rep, sz_rep = prep(va_rep)
print("групп sel/rep:", len(sz_sel), len(sz_rep),
      "| пересечение пользователей:", len(set(d_sel.uid) & set(d_rep.uid)))   # ожидается 0

# --- матрицы признаков (один раз) ---
XP = POOL2.to_numpy(dtype=np.float32)
CP = {c: i for i, c in enumerate(POOL2_COLS)}
Xtr_p  = XP[d_tr.index.to_numpy()]
Xsel_p = XP[d_sel.index.to_numpy()]
Xrep_p = XP[d_rep.index.to_numpy()]

def fit_sel(cols, seed=0, rounds=300, max_bin=63):
    ci = [CP[c] for c in cols]
    dtr  = lgb.Dataset(Xtr_p[:, ci], d_tr["label"].to_numpy(), group=sz_tr, params={"max_bin": max_bin})
    dsel = lgb.Dataset(Xsel_p[:, ci], d_sel["label"].to_numpy(), group=sz_sel, reference=dtr)
    m = lgb.train({**params_r, "seed": seed}, dtr, rounds, valid_sets=[dsel],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    p_sel = m.predict(Xsel_p[:, ci], num_iteration=m.best_iteration)
    p_rep = m.predict(Xrep_p[:, ci], num_iteration=m.best_iteration)
    s_sel = ndcg_at_k(d_sel["g"], p_sel, d_sel["label"])
    s_rep = ndcg_at_k(d_rep["g"], p_rep, d_rep["label"])
    del dtr, dsel; gc.collect()
    return m, s_sel, s_rep, p_rep

def ndcg_groups(groups, scores, labels, k=10, min_size=10):
    d = pd.DataFrame({"g": np.asarray(groups), "s": np.asarray(scores), "y": np.asarray(labels, dtype=float)})
    d = d[d.groupby("g")["y"].transform("size") >= min_size]
    a = d.sort_values(["g", "s"], ascending=[True, False]); ra = a.groupby("g").cumcount().to_numpy() + 1
    dcg = pd.Series(a["y"].to_numpy() / np.log2(ra + 1) * (ra <= k), index=a.index).groupby(a["g"]).sum()
    b = d.sort_values(["g", "y"], ascending=[True, False]); rb = b.groupby("g").cumcount().to_numpy() + 1
    idcg = pd.Series(b["y"].to_numpy() / np.log2(rb + 1) * (rb <= k), index=b.index).groupby(b["g"]).sum()
    return (dcg / idcg).dropna()

def boot_diff_rep(pred_a, pred_b, n_boot=1000, seed=0):
    ga = ndcg_groups(d_rep["g"], pred_a, d_rep["label"]); gb = ndcg_groups(d_rep["g"], pred_b, d_rep["label"])
    diff = (gb - ga).dropna()
    pu = diff.groupby(diff.index // 1000).agg(["sum", "count"])
    sums, cnts = pu["sum"].to_numpy(), pu["count"].to_numpy()
    r = np.random.default_rng(seed); est = []
    for _ in range(n_boot):
        idx = r.integers(0, len(sums), len(sums)); est.append(sums[idx].sum() / cnts[idx].sum())
    return diff.mean(), np.percentile(est, [2.5, 97.5])

# --- ориентиры на rep ---
rnd = ndcg_at_k(d_rep["g"], np.random.default_rng(0).random(len(d_rep)), d_rep["label"])
v = XP[d_rep.index.to_numpy(), CP["gitem_rate_30d"]]
pop = ndcg_at_k(d_rep["g"], np.where(np.isnan(v), 0.64, v), d_rep["label"])
print(f"rep: случайный {rnd:.4f} | популярность {pop:.4f}")

# --- M1, M2, пул ---
sets = {"M1": list(FEATS), "M2": list(M2_COLS), "POOL2": list(POOL2_COLS)}
R = {}
for name, cols in sets.items():
    m, s_sel, s_rep, p = fit_sel(cols)
    R[name] = dict(m=m, sel=s_sel, rep=s_rep, p=p, n=len(cols))
    print(f"{name:6s} ({len(cols)}): NDCG@10  sel={s_sel:.4f}  rep={s_rep:.4f}")

# --- A1 при K=26 и K=48 (отбор по важности из модели, обученной с остановкой по sel) ---
imp_s = pd.Series(R["POOL2"]["m"].feature_importance("gain"), index=POOL2_COLS).sort_values(ascending=False)
for K, ref in [(26, "M1"), (48, "M2")]:
    m, s_sel, s_rep, p = fit_sel(imp_s.index[:K].tolist())
    R[f"A1_K{K}"] = dict(m=m, sel=s_sel, rep=s_rep, p=p, n=K)
    mean, ci = boot_diff_rep(R[ref]["p"], p)
    print(f"A1 K={K}: sel={s_sel:.4f} rep={s_rep:.4f}; Δ(rep) к {ref} = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")
for ref, other in [("M1", "M2"), ("M2", "POOL2")]:
    mean, ci = boot_diff_rep(R[ref]["p"], R[other]["p"])
    print(f"{other} - {ref}: Δ(rep) = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")

In [ ]:
import matplotlib.pyplot as plt
Ks = [10, 15, 20, 26, 30, 40, 48, 60, 75, 97]
rows = []
for K in Ks:
    _, s_sel, s_rep, _ = fit_sel(imp_s.index[:K].tolist())
    rows.append((K, s_sel, s_rep)); print(K, round(s_sel, 4), round(s_rep, 4))
curve = pd.DataFrame(rows, columns=["K", "sel", "rep"])
plt.figure(figsize=(7, 4))
plt.plot(curve["K"], curve["rep"], "o-", label="A1 (отбор по важности), rep")
plt.scatter([26, 48], [R["M1"]["rep"], R["M2"]["rep"]], c="red", zorder=3, label="ручные M1 / M2")
plt.xlabel("число признаков K"); plt.ylabel("NDCG@10 (rep)"); plt.grid(alpha=.3); plt.legend()
plt.savefig("quality_vs_K.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
import json, time, warnings
warnings.filterwarnings("ignore", message="Overriding the parameters")

rnd_sel = ndcg_at_k(d_sel["g"], np.random.default_rng(0).random(len(d_sel)), d_sel["label"])
print(f"sel: случайный порядок {rnd_sel:.4f}")

# ---------- RFE ----------
def rfe_path(start_cols, targets, drop_frac=0.15):
    cols = list(start_cols); out = {}
    for K in sorted(targets, reverse=True):
        while len(cols) > K:
            m = fit_sel(cols)[0]
            imp = pd.Series(m.feature_importance("gain"), index=cols).sort_values(ascending=False)
            cols = imp.index[:max(K, int(len(cols) * (1 - drop_frac)))].tolist()
        out[K] = list(cols)
    return out

t = time.time()
sets_rfe = rfe_path(POOL2_COLS, [48, 26])
print("RFE готов за", round(time.time() - t), "с")

# ---------- A3: жадный прямой отбор на модели-прокси ----------
PROXY_FRAC, PROXY_TREES, PRESCREEN = 0.20, 60, 10
ru = np.random.default_rng(7)
tr_users = d_tr["uid"].unique()
u_sub = set(ru.choice(tr_users, size=int(len(tr_users) * PROXY_FRAC), replace=False).tolist())
m_sub = d_tr["uid"].isin(u_sub).to_numpy()
Xtr_sub = Xtr_p[m_sub]
y_sub = d_tr["label"].to_numpy()[m_sub]
sz_sub = d_tr.loc[m_sub].groupby("g", sort=True).size().to_numpy()
params_px = {**params_r, "learning_rate": 0.15, "num_leaves": 31, "seed": 0}

def proxy_fit(cols):
    ci = [CP[c] for c in cols]
    dtr = lgb.Dataset(Xtr_sub[:, ci], y_sub, group=sz_sub, params={"max_bin": 31})
    m = lgb.train(params_px, dtr, PROXY_TREES)
    p = m.predict(Xsel_p[:, ci])
    return m, ndcg_at_k(d_sel["g"], p, d_sel["label"])

def greedy_forward(all_cols, targets, prescreen=PRESCREEN, start=()):
    cur = list(start); out = {}; hist = []
    while len(cur) < max(targets):
        rest = [c for c in all_cols if c not in cur]
        m_all, _ = proxy_fit(cur + rest)                                  # прескрининг кандидатов
        g = pd.Series(m_all.feature_importance("gain"), index=cur + rest)[rest]
        cand = g.sort_values(ascending=False).index[:prescreen].tolist()
        scores = {c: proxy_fit(cur + [c])[1] for c in cand}
        best = max(scores, key=scores.get)
        cur.append(best)
        hist.append((len(cur), best, scores[best]))
        print(f"  K={len(cur):3d} + {best:20s} sel(proxy)={scores[best]:.4f}", flush=True)
        if len(cur) in targets:
            out[len(cur)] = list(cur)
    return out, hist

t = time.time()
sets_gr, hist_gr = greedy_forward(POOL2_COLS, [26, 48])
print("жадный отбор готов за", round((time.time() - t) / 60, 1), "мин")

In [ ]:
cands = {
    "A1":     {26: imp_s.index[:26].tolist(), 48: imp_s.index[:48].tolist()},
    "RFE":    sets_rfe,
    "Greedy": sets_gr,
}
rows = []
for K, ref in [(26, "M1"), (48, "M2")]:
    for meth, d in cands.items():
        m, s_sel, s_rep, p = fit_sel(d[K])
        R[f"{meth}_K{K}"] = dict(m=m, sel=s_sel, rep=s_rep, p=p, n=K)
        mean, ci = boot_diff_rep(R[ref]["p"], p)
        rows.append(dict(K=K, метод=meth, sel=s_sel, rep=s_rep, ref=ref,
                         d_ref=mean, ci_lo=ci[0], ci_hi=ci[1]))
        if meth != "A1":
            mean1, ci1 = boot_diff_rep(R[f"A1_K{K}"]["p"], p) if f"A1_K{K}" in R else (np.nan, (np.nan, np.nan))
            rows[-1].update(d_vs_A1=mean1, a1_lo=ci1[0], a1_hi=ci1[1])
print(pd.DataFrame(rows).round(4).to_string(index=False))

json.dump({m: {str(k): v for k, v in d.items()} for m, d in cands.items()},
          open("selected_sets_v1.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
import json, gc
SEEDS = [0, 1, 2, 3, 4]
FINALISTS = {
    "M1":     list(FEATS),
    "M2":     list(M2_COLS),
    "RFE_26": list(sets_rfe[26]),
    "RFE_48": list(sets_rfe[48]),
    "POOL2":  list(POOL2_COLS),
}
seed_rows = []
for name, cols in FINALISTS.items():
    for sd in SEEDS:
        _, s_sel, s_rep, _ = fit_sel(cols, seed=sd)
        seed_rows.append(dict(набор=name, seed=sd, sel=s_sel, rep=s_rep))
        print(name, sd, round(s_sel, 4), round(s_rep, 4), flush=True)
seed_df = pd.DataFrame(seed_rows)
print(seed_df.groupby("набор")[["sel", "rep"]].agg(["mean", "std"]).round(4))
seed_df.to_csv("seeds_default_params.csv", index=False)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 20
TUNE = ["M1", "M2", "RFE_26", "RFE_48"]

def fit_params(cols, hp, seed=0, rounds=400, max_bin=63):
    ci = [CP[c] for c in cols]
    dtr  = lgb.Dataset(Xtr_p[:, ci], d_tr["label"].to_numpy(), group=sz_tr, params={"max_bin": max_bin})
    dsel = lgb.Dataset(Xsel_p[:, ci], d_sel["label"].to_numpy(), group=sz_sel, reference=dtr)
    m = lgb.train({**params_r, **hp, "seed": seed}, dtr, rounds, valid_sets=[dsel],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    p_sel = m.predict(Xsel_p[:, ci], num_iteration=m.best_iteration)
    p_rep = m.predict(Xrep_p[:, ci], num_iteration=m.best_iteration)
    del dtr, dsel; gc.collect()
    return m, ndcg_at_k(d_sel["g"], p_sel, d_sel["label"]), ndcg_at_k(d_rep["g"], p_rep, d_rep["label"]), p_rep

DEFAULT_HP = dict(learning_rate=0.05, num_leaves=63, min_data_in_leaf=200,
                  feature_fraction=0.8, bagging_fraction=0.8, lambda_l2=1e-3)

def make_objective(cols):
    def objective(trial):
        hp = dict(
            learning_rate=trial.suggest_float("learning_rate", 0.03, 0.2, log=True),
            num_leaves=trial.suggest_int("num_leaves", 15, 255, log=True),
            min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 50, 1000, log=True),
            feature_fraction=trial.suggest_float("feature_fraction", 0.5, 1.0),
            bagging_fraction=trial.suggest_float("bagging_fraction", 0.5, 1.0),
            lambda_l2=trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
        )
        return fit_params(cols, hp)[1]               # решение принимается по sel
    return objective

best_hp = {}
for name in TUNE:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
    study.enqueue_trial(DEFAULT_HP)                   # первая попытка = текущие параметры
    study.optimize(make_objective(FINALISTS[name]), n_trials=N_TRIALS)
    best_hp[name] = study.best_params
    print(name, "лучший sel =", round(study.best_value, 4), study.best_params, flush=True)
json.dump(best_hp, open("best_hparams_v1.json", "w"), indent=1)

# оценка затюненных наборов на rep (3 seed)
tuned_rows, P_tuned = [], {}
for name in TUNE:
    for sd in [0, 1, 2]:
        _, s_sel, s_rep, p = fit_params(FINALISTS[name], best_hp[name], seed=sd)
        tuned_rows.append(dict(набор=name, seed=sd, sel=s_sel, rep=s_rep))
        if sd == 0:
            P_tuned[name] = p
tuned_df = pd.DataFrame(tuned_rows)
print(tuned_df.groupby("набор")[["sel", "rep"]].agg(["mean", "std"]).round(4))
for a, b in [("M1", "M2"), ("M1", "RFE_26"), ("M2", "RFE_48")]:
    mean, ci = boot_diff_rep(P_tuned[a], P_tuned[b])
    print(f"{b} - {a}: Δ(rep) = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")
tuned_df.to_csv("tuned_rep.csv", index=False)

In [ ]:
import json, time, re, os
df["skip"] = (df["pr"] < 0.1).astype(np.float32)
df["full"] = (df["pr"] >= 0.9).astype(np.float32)

KEYS = {"user": ["uid"], "item": ["item_id"], "ui": ["uid", "item_id"],
        "uart": ["uid", "artist_id"], "ualb": ["uid", "album_id"],
        "art": ["artist_id"], "alb": ["album_id"]}
AGGS = ["cnt", "rate", "mean_pr", "skip_rate", "full_rate", "recency"]
TARGET_OF = {"rate": "label", "mean_pr": "pr", "skip_rate": "skip", "full_rate": "full"}

def validate(p):
    try:
        key, agg = str(p["key"]), str(p["agg"])
        w = int(p.get("window", 1)) if agg != "recency" else 0
    except Exception:
        return None
    if key not in KEYS or agg not in AGGS:
        return None
    return {"key": key, "agg": agg, "window": min(max(w, 1), 400) if agg != "recency" else 0}

def fname(p):
    return f"{p['key']}_recency_d" if p["agg"] == "recency" else f"{p['key']}_{p['agg']}_{p['window']}d"

NEW = {}          # имя признака -> значения для всех строк df

def compile_programs(progs):
    groups = {}
    for p in progs:
        if fname(p) in NEW:
            continue
        kind = "recency" if p["agg"] == "recency" else ("cnt" if p["agg"] == "cnt" else TARGET_OF[p["agg"]])
        groups.setdefault((p["key"], kind), []).append(p)
    for (key, kind), ps in groups.items():
        keys = KEYS[key]
        if kind == "recency":
            F = add_window_feats2(df, keys, [], key, recency=True)
            NEW[f"{key}_recency_d"] = F[f"{key}_recency_d"].to_numpy()
        elif kind == "cnt":
            ws = sorted({p["window"] for p in ps})
            F = add_window_feats2(df, keys, ws, key, target="label", with_cnt=True)
            for w in ws:
                NEW[f"{key}_cnt_{w}d"] = F[f"{key}_cnt_{w}d"].to_numpy()
        else:
            ws = sorted({p["window"] for p in ps})
            tag = "rate" if kind == "label" else f"mean_{kind}"
            F = add_window_feats2(df, keys, ws, key, target=kind, with_cnt=False)
            for p in ps:
                NEW[fname(p)] = F[f"{key}_{tag}_{p['window']}d"].to_numpy()

# --- быстрая модель-прокси и индексы выборок ---
PROXY_FRAC, PROXY_TREES = 0.20, 60
ru = np.random.default_rng(7)
tr_users = d_tr["uid"].unique()
u_sub = set(ru.choice(tr_users, size=int(len(tr_users) * PROXY_FRAC), replace=False).tolist())
m_sub = d_tr["uid"].isin(u_sub).to_numpy()
y_sub = d_tr["label"].to_numpy()[m_sub]
sz_sub = d_tr.loc[m_sub].groupby("g", sort=True).size().to_numpy()
params_px = {**params_r, "learning_rate": 0.15, "num_leaves": 31, "seed": 0}
idx_tr_all, idx_sel, idx_rep = d_tr.index.to_numpy(), d_sel.index.to_numpy(), d_rep.index.to_numpy()
idx_tr_sub = idx_tr_all[m_sub]
ci_base = [CP[c] for c in FEATS]                       # базовый набор = M1

def proxy_score(Xtr, Xsel, seed=0):
    dtr = lgb.Dataset(Xtr, y_sub, group=sz_sub, params={"max_bin": 31})
    m = lgb.train({**params_px, "seed": seed}, dtr, PROXY_TREES)
    return ndcg_at_k(d_sel["g"], m.predict(Xsel), d_sel["label"])

Xb_tr0, Xb_sel0 = Xtr_p[m_sub][:, ci_base], Xsel_p[:, ci_base]
noise = [proxy_score(Xb_tr0, Xb_sel0, seed=s) for s in (0, 1, 2)]
THR = max(0.0005, 2 * float(np.std(noise)))            # порог принятия признака: 2 шума прокси
print(f"прокси на M1: {np.mean(noise):.4f} ± {np.std(noise):.4f}; порог принятия Δ > {THR:.4f}")

# --- поставщики предложений ---
def sample_program(rng):
    return validate({"key": str(rng.choice(list(KEYS))), "agg": str(rng.choice(AGGS)),
                     "window": int(np.exp(rng.uniform(0, np.log(400))))})

def propose_random(history, n, seen, rng):
    out = []
    while len(out) < n:
        p = sample_program(rng)
        if fname(p) not in seen and all(fname(p) != fname(q) for q in out):
            out.append(p)
    return out

def propose_evo(history, n, seen, rng):
    good = [h["p"] for h in history if h["delta"] > 0]
    out, tries = [], 0
    while len(out) < n and tries < 200:
        tries += 1
        if good and rng.random() < 0.6:                 # мутация удачной программы
            p = dict(good[int(rng.integers(len(good)))])
            r = rng.random()
            if r < 0.5 and p["agg"] != "recency":
                p["window"] = int(round(p["window"] * float(rng.choice([0.5, 0.7, 1.4, 2.0]))))
            elif r < 0.75:
                p["key"] = str(rng.choice(list(KEYS)))
            else:
                p["agg"] = str(rng.choice(AGGS))
            p = validate(p)
        else:
            p = sample_program(rng)
        if p and fname(p) not in seen and all(fname(p) != fname(q) for q in out):
            out.append(p)
    return out

PROMPT = """Ты помогаешь подбирать признаки для рекомендательной модели (ранжирование треков для пользователя).
Признак описывается ТОЛЬКО программой в формате JSON: {{"key": ..., "agg": ..., "window": число дней}}.
key: user (пользователь), item (трек), ui (пара пользователь-трек), uart (пользователь-исполнитель),
     ualb (пользователь-альбом), art (исполнитель), alb (альбом).
agg: cnt (число прошлых прослушиваний), rate (доля дослушанных >=50%), mean_pr (средняя доля прослушивания),
     skip_rate (доля скипов <10%), full_rate (доля почти полных >=90%), recency (дней с последнего прослушивания; window не нужен).
window: от 1 до 400 дней. Все признаки считаются только по прошлому, утечки невозможны.
Базовая модель уже использует 26 стандартных признаков (окна 1, 7, 30 дней для user/item/ui и часть item по всем пользователям).
Ниже история проверенных программ (delta = прирост качества NDCG@10, шум около {noise:.4f}).
{hist}
Предложи {n} НОВЫХ программ, которые, по твоему мнению, добавят больше всего информации. Не повторяй проверенные.
Ответь ТОЛЬКО JSON-массивом программ, без пояснений."""

def propose_llm(history, n, seen, rng=None):
    top, last = sorted(history, key=lambda h: -h["delta"])[:12], history[-12:]
    lines = [f'{json.dumps(h["p"])} -> delta={h["delta"]:+.4f}' for h in {id(x): x for x in top + last}.values()]
    prompt = PROMPT.format(noise=THR / 2, hist="\n".join(lines) if lines else "(пока пусто)", n=n)
    text = llm_call(prompt)                              # llm_call определяется отдельно (ячейка R)
    try:
        arr = json.loads(text[text.index("["): text.rindex("]") + 1])
    except Exception:
        return []
    out = []
    for item in arr:
        p = validate(item) if isinstance(item, dict) else None
        if p and fname(p) not in seen and all(fname(p) != fname(q) for q in out):
            out.append(p)
    return out[:n]

# --- цикл агента: раунд = предложение, оценка каждого кандидата, принятие лучшего ---
def run_agent(propose, rounds=10, per_round=8, seed=0):
    rng = np.random.default_rng(seed)
    Xtr_b, Xsel_b = Xb_tr0.copy(), Xb_sel0.copy()
    base = float(np.mean(noise))
    seen, accepted, history, t0 = set(FEATS), [], [], time.time()
    for r in range(rounds):
        progs = propose(history, per_round, seen, rng)
        if not progs:
            print(f"  раунд {r+1}: нет предложений"); continue
        compile_programs(progs)
        best = None
        for p in progs:
            nm = fname(p); seen.add(nm)
            s = proxy_score(np.column_stack([Xtr_b, NEW[nm][idx_tr_sub]]),
                            np.column_stack([Xsel_b, NEW[nm][idx_sel]]))
            history.append({"p": p, "delta": s - base})
            if best is None or s - base > best[1]:
                best = (p, s - base, s)
        if best[1] > THR:
            p, d_, s = best
            Xtr_b = np.column_stack([Xtr_b, NEW[fname(p)][idx_tr_sub]])
            Xsel_b = np.column_stack([Xsel_b, NEW[fname(p)][idx_sel]])
            accepted.append(fname(p)); base = s
        for p in progs:                                   # память: храним только принятые
            if fname(p) not in accepted:
                NEW.pop(fname(p), None)
        print(f"  раунд {r+1}: лучший {fname(best[0])} Δ={best[1]:+.4f}"
              f"{' (принят)' if best[1] > THR else ''}; принято {len(accepted)}", flush=True)
    return dict(accepted=accepted, history=history, evals=len(history), sec=time.time() - t0)

# --- итоговая оценка полной моделью: M1 + принятые признаки ---
def final_eval(names, seed=0):
    def X(Xp, idx): return np.column_stack([Xp[:, ci_base]] + [NEW[n][idx] for n in names])
    Xtr, Xs, Xr = X(Xtr_p, idx_tr_all), X(Xsel_p, idx_sel), X(Xrep_p, idx_rep)
    dtr = lgb.Dataset(Xtr, d_tr["label"].to_numpy(), group=sz_tr, params={"max_bin": 63})
    dsel = lgb.Dataset(Xs, d_sel["label"].to_numpy(), group=sz_sel, reference=dtr)
    m = lgb.train({**params_r, "seed": seed}, dtr, 300, valid_sets=[dsel],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    p_rep = m.predict(Xr, num_iteration=m.best_iteration)
    return ndcg_at_k(d_rep["g"], p_rep, d_rep["label"]), p_rep

In [ ]:
ROUNDS, PER_ROUND = 10, 8                 # бюджет: 80 оценок на метод
res_agents = {}
res_agents["random"] = run_agent(propose_random, ROUNDS, PER_ROUND, seed=0)
res_agents["evo"]    = run_agent(propose_evo,    ROUNDS, PER_ROUND, seed=0)
# после ячейки R (подключение LLM) добавьте:
# res_agents["llm"]  = run_agent(propose_llm,    ROUNDS, PER_ROUND, seed=0)

rows = []
for name, res in res_agents.items():
    s_rep, p = final_eval(res["accepted"])
    mean, ci = boot_diff_rep(R["M1"]["p"], p)
    rows.append(dict(метод=name, принято=len(res["accepted"]), оценок=res["evals"],
                     сек=round(res["sec"]), rep=s_rep, d_M1=mean, lo=ci[0], hi=ci[1]))
print(pd.DataFrame(rows).round(4).to_string(index=False))
json.dump({k: v["accepted"] for k, v in res_agents.items()}, open("agent_accepted_v1.json", "w"), indent=1)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def encode(p):
    v = [0.0] * (len(KEYS) + len(AGGS) + 1)
    v[list(KEYS).index(p["key"])] = 1.0
    v[len(KEYS) + AGGS.index(p["agg"])] = 1.0
    v[-1] = float(np.log1p(p["window"]))
    return v

def propose_surrogate(history, n, seen, rng, kappa=1.0, n_cand=300, warmup=16):
    if len(history) < warmup:                              # пока мало данных: случайный поиск
        return propose_random(history, n, seen, rng)
    X = np.array([encode(h["p"]) for h in history]); y = np.array([h["delta"] for h in history])
    rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
                               random_state=int(rng.integers(1_000_000))).fit(X, y)
    cands = propose_random(history, n_cand, seen, rng)     # кандидаты, которых ещё не проверяли
    Xc = np.array([encode(p) for p in cands])
    preds = np.stack([t.predict(Xc) for t in rf.estimators_])
    score = preds.mean(0) + kappa * preds.std(0)           # ожидаемый прирост + бонус за неопределённость
    return [cands[i] for i in np.argsort(-score)[:n]]

In [ ]:
SEEDS_A = [0, 1, 2]
multi = {}
for name, prop in [("random", propose_random), ("evo", propose_evo), ("surrogate", propose_surrogate)]:
    multi[name] = []
    for s in SEEDS_A:
        print(name, "seed", s, flush=True)
        multi[name].append(run_agent(prop, ROUNDS, PER_ROUND, seed=s))

rows = []
for name, runs in multi.items():
    for s, res in zip(SEEDS_A, runs):
        s_rep, p = final_eval(res["accepted"])
        mean, ci = boot_diff_rep(R["M1"]["p"], p)
        first = next((i + 1 for i, h in enumerate(res["history"]) if h["delta"] > THR), None)
        rows.append(dict(метод=name, seed=s, принято=len(res["accepted"]), оценок_до_первого=first,
                         rep=s_rep, d_M1=mean, сек=round(res["sec"])))
summary = pd.DataFrame(rows)
print(summary.round(4).to_string(index=False))
print(summary.groupby("метод")[["принято", "rep", "d_M1"]].agg(["mean", "std"]).round(4))

plt.figure(figsize=(7, 4))
for name, runs in multi.items():
    curves = [np.maximum.accumulate([h["delta"] for h in res["history"]]) for res in runs]
    L = min(len(c) for c in curves)
    plt.plot(range(1, L + 1), np.mean([c[:L] for c in curves], axis=0), label=name)
plt.axhline(THR, color="gray", ls="--", label="порог принятия")
plt.xlabel("число оценённых признаков (бюджет)"); plt.ylabel("лучший прирост Δ (прокси, sel)")
plt.grid(alpha=.3); plt.legend()
plt.savefig("agent_curves.png", dpi=150, bbox_inches="tight"); plt.show()
json.dump({k: [r["accepted"] for r in v] for k, v in multi.items()}, open("agent_accepted_multi.json", "w"), indent=1)

In [ ]:
acc_names = sorted({n for runs in multi.values() for res in runs for n in res["accepted"]})
print(len(acc_names), "принятых признаков:", acc_names)
for nm in acc_names:
    if nm in CP:
        continue
    CP[nm] = Xtr_p.shape[1]
    Xtr_p  = np.column_stack([Xtr_p,  NEW[nm][idx_tr_all]])
    Xsel_p = np.column_stack([Xsel_p, NEW[nm][idx_sel]])
    Xrep_p = np.column_stack([Xrep_p, NEW[nm][idx_rep]])
POOL3_COLS = list(POOL2_COLS) + [n for n in acc_names if n not in POOL2_COLS]
print(len(POOL3_COLS), "признаков в пуле v3")

Ks3 = [48, 26]
t = time.time()
sets_rfe3 = rfe_path(POOL3_COLS, Ks3)
print("RFE на пуле v3:", round(time.time() - t), "с")
for K in Ks3:
    for tag, cols in [("RFE v2", sets_rfe[K]), ("RFE v3", sets_rfe3[K])]:
        _, s_sel, s_rep, p = fit_sel(cols)
        R[f"{tag}_{K}"] = dict(p=p, rep=s_rep)
        print(K, tag, round(s_sel, 4), round(s_rep, 4))
    mean, ci = boot_diff_rep(R[f"RFE v2_{K}"]["p"], R[f"RFE v3_{K}"]["p"])
    print(f"K={K}: RFE v3 - RFE v2 = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")

In [ ]:
import json

# Путь к сохранённому json-файлу
multi_path = '/kaggle/input/notebooks/esgrimmm/notebooka462905bc3/agent_accepted_multi.json'

with open(multi_path, 'r', encoding='utf-8') as f:
    multi = json.load(f)

print("Данные 'multi' успешно загружены в память!")

In [ ]:
for name, prop in [("random", propose_random), ("evo", propose_evo), ("surrogate", propose_surrogate)]:
    for s in range(3, 10):
        multi[name].append(run_agent(prop, ROUNDS, PER_ROUND, seed=s))
SEEDS_A = list(range(10))
# затем перезапустите часть ячейки S, начиная с формирования таблицы summary (без повторного запуска агентов)

In [ ]:
from huggingface_hub import get_hf_file_metadata, hf_hub_url
meta = get_hf_file_metadata(hf_hub_url("yandex/yambda", "embeddings.parquet", repo_type="dataset"))
print(round(meta.size / 1e9, 2), "ГБ")

## 8. Графовые признаки

Проверка альтернативных представлений на основе двудольного графа пользователь–трек.

In [ ]:
import scipy.sparse as sp
from scipy.sparse.linalg import svds

def spectral_item_vectors(u, i, d=16, min_deg=5):
    """Вложения треков из двудольного графа пользователь-трек (усечённое SVD нормированной матрицы)."""
    ucodes, uidx = pd.factorize(u); icodes, iidx = pd.factorize(i)
    M = sp.coo_matrix((np.ones(len(ucodes), np.float32), (ucodes, icodes)),
                      shape=(len(uidx), len(iidx))).tocsr()
    M.data = np.minimum(M.data, 1.0)                              # бинаризация
    deg_i = np.asarray(M.sum(0)).ravel(); keep = deg_i >= min_deg
    M, iidx = M[:, keep], iidx[keep]
    du = np.asarray(M.sum(1)).ravel(); di = np.asarray(M.sum(0)).ravel()
    M = sp.diags(1 / np.sqrt(np.maximum(du, 1))) @ M @ sp.diags(1 / np.sqrt(np.maximum(di, 1)))
    _, S, Vt = svds(M.astype(np.float64), k=d)
    V = Vt.T * S
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    return pd.Series(np.arange(len(iidx)), index=pd.Index(iidx)), V.astype(np.float32)

def taste_feats(df, vec_index, V, windows_days, name, snap=DAY, chunk=1_000_000):
    """Косинус между вектором кандидата и 'вектором вкуса' пользователя: средним векторов дослушанных (pos)
    и скипнутых (neg) треков из прошлого в окне. Только по прошлому, снимок на начало суток."""
    n, d = len(df), V.shape[1]
    pos = vec_index.reindex(df["item_id"].to_numpy()).to_numpy(dtype=np.float64)
    has = ~np.isnan(pos)
    E = V[np.where(has, pos, 0).astype(np.int64)].astype(np.float32)
    E[~has] = 0.0
    codes = df.groupby("uid", sort=False).ngroup().to_numpy(np.int64)
    ts = df["timestamp"].to_numpy(np.int64)
    tq = (ts // snap) * snap
    comb = (codes << SHIFT) | ts
    order = np.argsort(comb, kind="stable")
    comb_s, codes_s, tq_s = comb[order], codes[order], tq[order]
    E_s, has_s = E[order], has[order]
    lab_s = df["label"].to_numpy()[order].astype(np.float64)
    hi = np.searchsorted(comb_s, (codes_s << SHIFT) | tq_s, side="left")
    los = {w: np.searchsorted(comb_s, (codes_s << SHIFT) | np.maximum(tq_s - w * DAY, 0), side="left")
           for w in windows_days}
    out = {}
    for sign, wt in (("pos", lab_s), ("neg", 1.0 - lab_s)):
        wt = wt * has_s
        cumw = np.concatenate([[0.0], np.cumsum(wt)])
        cumv = np.zeros((n + 1, d), np.float64)
        np.cumsum(E_s * wt[:, None], axis=0, out=cumv[1:])
        for w in windows_days:
            res_s = np.full(n, np.nan, np.float32)
            lo = los[w]
            for a in range(0, n, chunk):
                b = min(n, a + chunk)
                S = cumv[hi[a:b]] - cumv[lo[a:b]]
                W = cumw[hi[a:b]] - cumw[lo[a:b]]
                nrm = np.linalg.norm(S, axis=1)
                dot = np.einsum("ij,ij->i", S, E_s[a:b].astype(np.float64))
                ok = (W > 0) & (nrm > 1e-9) & has_s[a:b]
                res_s[a:b] = np.where(ok, dot / np.maximum(nrm, 1e-9), np.nan)
            r = np.empty(n, np.float32); r[order] = res_s
            out[f"{name}_taste_{sign}_{w}d"] = r
        del cumv
    res = pd.DataFrame(out, index=df.index)
    for w in windows_days:
        res[f"{name}_taste_diff_{w}d"] = (res[f"{name}_taste_pos_{w}d"] - res[f"{name}_taste_neg_{w}d"]).astype(np.float32)
    return res

In [ ]:
T0 = int(t_val - TRAIN_DAYS * DAY)       # граф строится только по событиям СТРОГО до начала окна обучения
tab = pq.read_table(path, columns=["uid", "item_id", "timestamp", "played_ratio_pct"],
                    filters=[("event_type", "==", "listen"), ("timestamp", "<", T0),
                             ("played_ratio_pct", ">=", POS_THRESHOLD)])
past = tab.to_pandas(); del tab
print(len(past), "дослушанных прослушиваний до T0")
vec_index, V_graph = spectral_item_vectors(past["uid"].to_numpy(), past["item_id"].to_numpy(), d=16, min_deg=5)
del past
print("вложений треков:", V_graph.shape, "| покрытие строк df:", round(df["item_id"].isin(vec_index.index).mean(), 3))

TASTE = taste_feats(df, vec_index, V_graph, [30, 90, 400], "graph")
print(TASTE.notna().mean().round(3))

# быстрая проверка сигнала: NDCG@10 одной фичи на rep (случайный порядок 0,6950; популярность 0,7226)
for c in ["graph_taste_pos_90d", "graph_taste_neg_90d", "graph_taste_diff_90d"]:
    v = TASTE[c].to_numpy()[idx_rep]
    print(c, round(ndcg_at_k(d_rep["g"], np.nan_to_num(v, nan=np.nanmean(v)), d_rep["label"]), 4))